In [ ]:
import os
from dotenv import load_dotenv

load_dotenv() 

token = os.getenv("TABPFN_TOKEN")

In [ ]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error

from tabpfn import TabPFNRegressor


df = pd.read_csv(r"C:\Users\Mayank\Downloads\TabPFN-experiments\datasets\msedcl_reliability_dataset\msedcl_reliability.csv")

target = "saifi"  # or "saidi_min", "caidi_min"

df = df.dropna(subset=[target])
df["month"] = pd.to_datetime(df["report_month"]).dt.month
df["circle_id"] = df["circle"].astype("category").cat.codes
df["value_3m_ago"] = df.groupby("circle")[target].shift(3)
df["value_1y_ago"] = df.groupby("circle")[target].shift(12)
df = df.dropna(subset=["value_3m_ago", "value_1y_ago"]).sort_values("report_month")

X = df[["circle_id", "month", "value_3m_ago", "value_1y_ago"]]
y = df[target]


X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=500,
    shuffle=False
)

X_train = X_train.iloc[-3000:]
y_train = y_train.iloc[-3000:]

model = TabPFNRegressor(device="cuda")


model.fit(X_train, y_train)


preds = model.predict(X_val)


mae = mean_absolute_error(y_val, preds)
wape = abs(y_val - preds).sum() / y_val.sum() * 100

print("MAE in predicting", target, mae)
print("WAPE in predicting", target, wape, "%")